In [ ]:
import time
import matplotlib.pyplot as plt

# Calculate appropriate threshold (average dollar volume per tick * desired ticks per bar)
avg_dollar_volume = (btc_pl['price'] * btc_pl['volume']).mean()
desired_ticks_per_bar = 100
dollar_threshold = avg_dollar_volume * desired_ticks_per_bar

print(f"Using dollar threshold: ${dollar_threshold:,.2f}")

# Create dollar bars
start_time = time.time()
dollar_bars_pl = create_dollar_bars_polars(btc_pl.select(['timestamp', 'price', 'volume']), dollar_threshold)
polars_time = time.time() - start_time

print(f"\nCreated {len(dollar_bars_pl)} dollar bars in {polars_time:.3f} seconds")
print(f"\nFirst few dollar bars:")
print(dollar_bars_pl.head())


In [ ]:
# Create line plot of price data
plt.figure(figsize=(12, 6))
plt.plot(dollar_bars_pl['close'], linewidth=1)
plt.title('BTC Price vs Bar Number')
plt.xlabel('Bar Number')
plt.ylabel('Price')
plt.grid(True)
plt.show()
from pathlib import Path
import polars as pl

In [ ]:
DATA_PATH = Path('..') / 'dev' / 'data' / 'binance_test'

In [ ]:
btc_pl = pl.read_parquet(DATA_PATH / 'btcusdt')


In [ ]:
btc_pl.shape

In [ ]:
def create_dollar_bars_polars(df: pl.DataFrame, dollar_threshold: float) -> pl.DataFrame:
    """
    Create dollar bars from tick data using Polars.

    Args:
        df: Polars DataFrame with columns ['timestamp', 'price', 'volume']
        dollar_threshold: Dollar volume threshold for each bar

    Returns:
        DataFrame with OHLCV dollar bars
    """
    # Calculate dollar volume for each tick
    df = df.with_columns(
        (pl.col('price') * pl.col('volume')).alias('dollar_volume')
    )

    # Calculate cumulative dollar volume (use cum_sum instead of cumsum)
    df = df.with_columns(
        pl.col('dollar_volume').cum_sum().alias('cum_dollar_volume')
    )

    # Identify bar boundaries (when threshold is exceeded)
    df = df.with_columns(
        (pl.col('cum_dollar_volume') // dollar_threshold).alias('bar_id')
    )

    # Aggregate to create bars
    dollar_bars = df.group_by('bar_id').agg([
        pl.col('timestamp').first().alias('timestamp'),
        pl.col('price').first().alias('open'),
        pl.col('price').max().alias('high'),
        pl.col('price').min().alias('low'),
        pl.col('price').last().alias('close'),
        pl.col('volume').sum().alias('volume'),
        pl.col('dollar_volume').sum().alias('dollar_volume'),
        pl.len().alias('tick_count'),
        pl.col('price').std().alias('price_std'),
        (pl.col('price').last() - pl.col('price').first()).alias('price_change')
    ]).sort('bar_id')

    return dollar_bars


The code failed because:
- The DataFrame uses 'quantity' instead of 'volume', and 'price' is a string type that must be cast to numeric.
- The input parquet path likely needs a file (e.g., 'btcusdt.parquet') rather than a directory.
- The function expects ['timestamp', 'price', 'volume']; we need to adapt it to existing columns and types.

Fixed:
- Read the correct parquet file.
- Cast 'price' and 'quantity' to Float64 and rename 'quantity' -> 'volume'.
- Use the adapted DataFrame with create_dollar_bars_polars.

In [ ]:
# Ensure correct file path and schema for btc_pl
btc_pl = pl.read_parquet(DATA_PATH / 'btcusdt')\
    .with_columns([
        pl.col('price').cast(pl.Float64),
        pl.col('quantity').cast(pl.Float64).alias('volume')
    ])

bars_pl = create_dollar_bars_polars(df=btc_pl.select(['timestamp', 'price', 'volume']), dollar_threshold=100_000)


In [ ]:
bars_pl.shape

In [ ]:

# Calculate appropriate threshold (average dollar volume per tick * desired ticks per bar)
avg_dollar_volume = (tick_data['price'] * tick_data['volume']).mean()
desired_ticks_per_bar = 100
dollar_threshold = avg_dollar_volume * desired_ticks_per_bar

print(f"Using dollar threshold: ${dollar_threshold:,.2f}")

# Create dollar bars
start_time = time.time()
dollar_bars_pl = create_dollar_bars_polars(tick_data_pl, dollar_threshold)
polars_time = time.time() - start_time

print(f"\nCreated {len(dollar_bars_pl)} dollar bars in {polars_time:.3f} seconds")
print(f"\nFirst few dollar bars:")
print(dollar_bars_pl.head())